<a href="https://colab.research.google.com/github/dhag/colab_demo/blob/main/gemini_colab_mcp_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab で MCP を使うサンプル（Gemini 連携）

**MCP (Model Context Protocol)** を Colab 上で動かし、Gemini から外部ツールを呼び出すデモです。

## このノートの位置づけ（重要）
- **MCP が目指す姿 ＝ 全自動**：AI に MCP サーバーを「つなぐだけ」で、AI が自分でツールを見つけて呼ぶ。
  人間はツールを書くだけでよい（USB-C のように挿すだけ、という理想）。
- **2026年7月の現状 ＝ まだ手動**：この自動連携はまだ実験的で、現行 SDK では動かない。
  そこで今は、本来 SDK が自動でやってくれる処理（ツールの変換と呼び出しループ）を **手動で** 書く。
- **近い未来 ＝ 自動に戻る**：SDK が対応すれば手動部分は消え、「つなぐだけ」で全自動になる。

進め方：**①理想の書き方を試して「今はまだ動かない」ことを確認 → ②現状の手動の書き方で動かす**。
手動版は中身（ツール発見・呼び出しループ）が見えるので、MCP の仕組みの理解にも向いています。

## Colab 特有の注意（詳しくは末尾のトラブルシューティング）
1. **イベントループ**：Colab は既にループが動いているため、別スレッドの独立ループ（`run_async`）で実行する。
2. **stdio トランスポート**：サーバーはサブプロセスとして起動する（Node.js 不要にするため Python で自作）。
3. **標準エラーの fileno**：Colab の `sys.stderr` には `fileno()` が無いので、サーバーの stderr を `/dev/null` へ逃がす。


## 1. ライブラリのインストール

In [1]:
!pip install -q google-genai mcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.1 MB/s eta 0:00:00


## 2. API キーの設定

Colab 左の 🔑（シークレット）に `GEMINI_API_KEY` を登録しておくのが安全です。無ければ手入力します。

In [2]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("Colab シークレットから API キーを読み込みました")
except Exception:
    import getpass
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY を入力: ")
    print("手入力で API キーを設定しました")

Colab シークレットから API キーを読み込みました


## 3. MCP サーバーを作る

`%%writefile` でサーバーを別ファイルに保存します（サブプロセスとして起動するため）。
`FastMCP` を使うと、関数に `@mcp.tool()` を付けるだけでツールとして公開できます。
docstring と型ヒントがそのままツールの説明・引数仕様になるので、**分かりやすい docstring** が
AI の呼び出し精度を左右します。

## 本来、MCPサーバはコンピュータ内で実行する処理（ファイルのアクセスやアプリの実行など危険な処理を含む）を行う。ここでは雑なシミュレーションで。

In [3]:
%%writefile my_mcp_server.py
"""教育デモ用の最小 MCP サーバー。stdio で通信する。"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo-server")


@mcp.tool()
def add(a: float, b: float) -> float:
    """2つの数値を加算して返す。

    Args:
        a: 1つ目の数値
        b: 2つ目の数値
    """
    return a + b


@mcp.tool()
def multiply(a: float, b: float) -> float:
    """2つの数値を乗算して返す。

    Args:
        a: 1つ目の数値
        b: 2つ目の数値
    """
    return a * b


@mcp.tool()
def get_weather(city: str) -> str:
    """指定した都市の天気を返す（デモ用のダミーデータ）。

    Args:
        city: 都市名（例: 盛岡, 東京）
    """
    db = {
        "盛岡": "曇りときどき晴れ、最高18度",
        "東京": "晴れ、最高25度",
    }
    return db.get(city.strip(), f"{city} のデータはありません（デモDB未登録）")


@mcp.tool()
def next_class(day: str) -> str:
    """指定した曜日の講義予定を返す（デモ用の固定データ）。

    Args:
        day: 曜日。「木」「木曜」「木曜日」のいずれの形でも指定できる。
    """
    timetable = {
        "月": "3限: 信号処理",
        "火": "2限: ニューラルネットワーク",
        "水": "休講",
        "木": "4限: 制御工学",
        "金": "1限: 電子回路",
    }
    # 「木曜日」「木曜」「木」などを先頭一文字（曜日）に正規化する
    key = day.strip().replace("曜日", "").replace("曜", "")[:1]
    return timetable.get(key, f"{day} の予定は登録されていません")


if __name__ == "__main__":
    mcp.run(transport="stdio")


Writing my_mcp_server.py


## 4. 共通の準備（クライアントと実行ヘルパ）

Gemini クライアント、モデル名、サーバー起動設定、そして非同期を安全に回すための
`run_async`（別スレッドの独立イベントループ）を用意します。理想版・手動版の両方でこれらを使います。

In [4]:
import os
import asyncio
import threading

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from google import genai
from google.genai import types

client = genai.Client()  # GEMINI_API_KEY を環境変数から自動取得

# 使うモデル。古くなっていたら次のセルの一覧から選んで置き換えてください。
MODEL = "gemini-2.5-flash"

# 自作 MCP サーバーを Python サブプロセスとして起動する設定
server_params = StdioServerParameters(
    command="python",
    args=["my_mcp_server.py"],
    env=None,
)

# Colab の sys.stderr には fileno() が無く、そのままだとサーバー起動時に落ちる。
# サーバーの標準エラー出力を実ファイル(/dev/null)へ逃がして回避する。
_DEVNULL = open(os.devnull, "w")


# Colab の既存ループに nest_asyncio でネストすると MCP 内部（anyio）が壊れやすい。
# 干渉しない「別スレッドの独立イベントループ」でコルーチンを実行する。
def run_async(coro):
    box = {}
    def worker():
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except BaseException as e:
            box["error"] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker)
    t.start()
    t.join()
    if "error" in box:
        raise box["error"]
    return box["value"]

## 5. 使えるモデルを確認（任意）

`gemini-2.5-flash` が使えない環境なら、下の一覧から `generateContent` 対応モデルを選んで
上の `MODEL` に設定してください。

In [5]:
for m in client.models.list():
    actions = getattr(m, "supported_actions", None) or []
    if (not actions) or ("generateContent" in actions):
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

## 6.【理想】つなぐだけで全自動 —— MCP が目指す姿

MCP の狙いは、**AI にサーバーを「つなぐだけ」で、ツールの発見・選択・実行・結果の反映まで
SDK が全部自動でやってくれる**ことです。理想の書き方はこれだけ 👇

```python
config = types.GenerateContentConfig(tools=[session])  # ← セッションを渡すだけ
```

ところが **2026年7月時点の `google-genai` ではこの自動連携がまだ実験的**で、
SDK が内部設定をコピーする際に MCP セッションを扱えず失敗します。
下のセルで実際に試して、「今はまだ動かない」ことを確認しましょう
（近い将来 SDK が対応すれば、この書き方だけで動くようになります）。

In [6]:
async def ask_auto(prompt: str) -> str:
    # 【理想】セッションをそのまま tools に渡す＝全自動。今はまだ動かない。
    async with stdio_client(server_params, errlog=_DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            response = await client.aio.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(tools=[session]),
            )
            return response.text


# 実際に試す（現時点では失敗する見込み）
try:
    print(run_async(ask_auto("2 と 3 を足して")))
except Exception as e:
    print("いまは自動連携が失敗します:", type(e).__name__)
    print("→ 原因は SDK が設定をコピーする際に MCP セッションを扱えないため。")
    print("→ 次の『手動版』を使います（将来 SDK が対応すればこのセルだけで動きます）。")

いまは自動連携が失敗します: ExceptionGroup
→ 原因は SDK が設定をコピーする際に MCP セッションを扱えないため。
→ 次の『手動版』を使います（将来 SDK が対応すればこのセルだけで動きます）。


## 7.【現状】いまは手動でツールを変換してループを回す

自動連携が動くようになるまでは、**本来 SDK が裏でやってくれる処理を手で書きます**。
やることは3つだけです。

1. MCP のツール一覧を Gemini の**関数宣言**に変換する（セッションは渡さない）
2. Gemini に「どのツールを呼ぶか」決めさせる
3. 決まったツールを MCP サーバーで実行し、結果を返してもう一度生成する（これを繰り返す）

> SDK が自動連携に対応すれば、この節はまるごと不要になり、## 6 の書き方だけで済むようになります。
> 逆に今は、この手動版のおかげで「ツール発見 → 呼び出し → 結果反映」の中身が見えます。

In [7]:
def _clean_schema(schema):
    """MCP の inputSchema(JSON Schema) から Gemini が受け付けないキー(title 等)を除去する。"""
    if not isinstance(schema, dict):
        return schema
    drop = {"title", "$schema", "additionalProperties", "default"}
    out = {}
    for k, v in schema.items():
        if k in drop:
            continue
        if k == "properties" and isinstance(v, dict):
            out[k] = {pk: _clean_schema(pv) for pk, pv in v.items()}
        elif k == "items" and isinstance(v, dict):
            out[k] = _clean_schema(v)
        else:
            out[k] = v
    return out


async def ask(prompt: str) -> str:
    async with stdio_client(server_params, errlog=_DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # 1) MCP のツール一覧 → Gemini の関数宣言に変換
            listed = await session.list_tools()
            decls = [
                types.FunctionDeclaration(
                    name=t.name,
                    description=(t.description or "").splitlines()[0],
                    parameters=_clean_schema(t.inputSchema),
                )
                for t in listed.tools
            ]
            config = types.GenerateContentConfig(
                temperature=0,
                tools=[types.Tool(function_declarations=decls)],
            )

            # 2) function calling ループを手動で回す
            contents = [types.Content(role="user", parts=[types.Part(text=prompt)])]
            for _ in range(5):  # 最大5往復（無限ループ防止）
                response = await client.aio.models.generate_content(
                    model=MODEL, contents=contents, config=config,
                )
                parts = response.candidates[0].content.parts or []
                calls = [p.function_call for p in parts if p.function_call]
                if not calls:
                    return response.text  # ツール呼び出しが無ければ最終回答

                contents.append(response.candidates[0].content)  # モデルの呼び出し要求を履歴へ

                # 3) 各ツールを MCP サーバーで実行し、結果を function_response で返す
                result_parts = []
                for call in calls:
                    r = await session.call_tool(call.name, dict(call.args or {}))
                    text = r.content[0].text if r.content else ""
                    result_parts.append(
                        types.Part.from_function_response(
                            name=call.name, response={"result": text}
                        )
                    )
                contents.append(types.Content(role="user", parts=result_parts))

            return response.text

## 8. 実行してみる

Gemini が質問に応じて自動でツールを選び、手動ループが実行します。

In [8]:
print(run_async(ask("盛岡の天気は？あと 123 と 456 を足すといくつ？")))

盛岡の天気は「曇りときどき晴れ、最高18度」です。
123 と 456 を足すと 579 になります。


In [9]:
print(run_async(ask("木曜の講義は何限にある？その講義名も教えて。")))

木曜の講義は4限に制御工学があります。


In [10]:
print(run_async(ask("7 と 8 を掛けて、その結果に 10 を足して。")))

7 と 8 を掛けた結果に 10 を足すと 66 になります。


## 9.（おまけ）SDK に任せず手動で MCP を触る

「どのツールが公開されているか」「AI を介さず直接ツールを叩く」を確認したいとき用。
MCP の仕組みそのものを見せたい場合に便利です。

In [11]:
async def inspect_server():
    async with stdio_client(server_params, errlog=_DEVNULL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("=== 公開ツール一覧 ===")
            for t in tools.tools:
                print(f"- {t.name}: {t.description.splitlines()[0]}")

            print("\n=== 直接呼び出し: add(2, 3) ===")
            result = await session.call_tool("add", {"a": 2, "b": 3})
            print(result.content[0].text)

run_async(inspect_server())

=== 公開ツール一覧 ===
- add: 2つの数値を加算して返す。
- multiply: 2つの数値を乗算して返す。
- get_weather: 指定した都市の天気を返す（デモ用のダミーデータ）。
- next_class: 指定した曜日の講義予定を返す（デモ用の固定データ）。

=== 直接呼び出し: add(2, 3) ===
5.0


## まとめ

- **現状は手動**：`tools=[session]` の自動連携は 2026年7月時点の SDK ではまだ動かない（設定の内部コピーで失敗）。
  そこで今は、ツールを関数宣言に変換して呼び出しループを手で回している。
- **近い未来は自動**：SDK が対応すれば ## 7 の手動部分は不要になり、## 6 の「つなぐだけ」で全自動になる。
- MCP はプロトコルとして策定済みだが、各 SDK の実装が追いつく**過渡期**にある。
  手動版は、その間の橋渡しであると同時に、function calling の中身を学べる教材でもある。

## トラブルシューティング
- **`io.UnsupportedOperation: fileno`**：Colab の `sys.stderr` に `fileno()` が無いのが原因。
  サーバーの stderr を `/dev/null` へ逃がす（`errlog=open(os.devnull, "w")`。本ノートは対応済み）。
- **`ExceptionGroup` / サブ例外が読めない**：`nest_asyncio` で Colab のループにネストすると MCP 内部が壊れる。
  本ノートのように `run_async`（別スレッドの独立ループ）で実行する。
- **`generate_content` で `model_copy` / `deepcopy` 系エラー**：`tools=[session]`（自動連携）を使ったときに出る、
  今回のメインの制約。手動版（## 7）に切り替える。
- **ツールが呼ばれない / 引数がずれる**：docstring と引数名を具体的にする。サーバー側で引数を正規化しておくと堅牢
  （例：`next_class` は「木曜」「木曜日」を「木」に直している）。
- **モデル名エラー**：`MODEL` を ## 5 の一覧にあるモデルに置き換える。
